In [3]:
import optuna
from tensorflow import keras
from keras import layers
from sklearn.metrics import roc_auc_score
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [4]:
df = pd.read_csv('/home/prateek/Prateek/LaunchPad/week6/Day3/src/data/processed/final.csv')

X = df.drop('Survived', axis=1).values
y = df['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/x_train.npy', X_train)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/x_test.npy', X_test)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/y_train.npy', y_train)
np.save('/home/prateek/Prateek/LaunchPad/week6/Day4/src/data/processed/y_test.npy', y_test)


(712, 5) (179, 5)


In [5]:
def build_model(trial, input_dim):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for i in range(n_layers):
        units = trial.suggest_int(f"units_{i}", 16, 128, step=16)
        dropout = trial.suggest_float(f"dropout_{i}", 0.1, 0.5)
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(1, activation="sigmoid"))

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [6]:
def make_objective(X_train, y_train, X_test, y_test):
    def objective(trial):
        model = build_model(trial, X_train.shape[1])

        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

        early_stopping = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )

        model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=100,
            batch_size=batch_size,
            callbacks=[early_stopping],
            verbose=0
        )

        y_pred_proba = model.predict(X_test).flatten()
        auc = roc_auc_score(y_test, y_pred_proba)

        return auc

    return objective


In [ ]:
objective_fn = make_objective(X_train, y_train, X_test, y_test)
study = optuna.create_study(direction="maximize")
study.optimize(objective_fn, n_trials=50)

[I 2026-01-29 13:25:14,116] A new study created in memory with name: no-name-99a282ae-4b63-44c5-9ac8-58efeaf40ff1
2026-01-29 13:25:14.122048: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 


[I 2026-01-29 13:25:22,828] Trial 0 finished with value: 0.8326086956521739 and parameters: {'n_layers': 2, 'units_0': 80, 'dropout_0': 0.4450221862437729, 'units_1': 80, 'dropout_1': 0.3209257623365671, 'lr': 0.000102540922698182, 'batch_size': 32}. Best is trial 0 with value: 0.8326086956521739.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-29 13:25:32,153] Trial 1 finished with value: 0.8410408432147563 and parameters: {'n_layers': 1, 'units_0': 96, 'dropout_0': 0.20633212296603448, 'lr': 0.00028569311969094965, 'batch_size': 16}. Best is trial 1 with value: 0.8410408432147563.


...

[I 2026-01-29 13:29:10,661] Trial 47 finished with value: 0.8440711462450593 and parameters: {'n_layers': 1, 'units_0': 48, 'dropout_0': 0.41842557519865176, 'lr': 0.005556686362736907, 'batch_size': 16}. Best is trial 23 with value: 0.8555335968379446.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-29 13:29:20,110] Trial 48 finished with value: 0.8451251646903821 and parameters: {'n_layers': 1, 'units_0': 32, 'dropout_0': 0.1786259344057795, 'lr': 0.0007971703151956552, 'batch_size': 16}. Best is trial 23 with value: 0.8555335968379446.


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


[I 2026-01-29 13:29:23,183] Trial 49 finished with value: 0.8452569169960474 and parameters: {'n_layers': 1, 'units_0': 48, 'dropout_0': 0.35548128661772754, 'lr': 0.006841720873512572, 'batch_size': 16}. Best is trial 23 with value: 0.8555335968379446.


In [8]:
print("Best AUC:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")


Best AUC: 0.8555335968379446
Best params:
n_layers: 3
units_0: 48
dropout_0: 0.4155033456064326
units_1: 112
dropout_1: 0.10091032320655559
units_2: 112
dropout_2: 0.48068067271530523
lr: 0.0015058534245549293
batch_size: 16


In [9]:
from keras.callbacks import EarlyStopping
best_params = study.best_params

def build_best_model(input_dim):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for i in range(best_params["n_layers"]):
        model.add(layers.Dense(
            best_params[f"units_{i}"],
            activation="relu"
        ))
        model.add(layers.Dropout(best_params[f"dropout_{i}"]))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(best_params["lr"]),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

final_model = build_best_model(X_train.shape[1])

final_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=best_params["batch_size"],
    validation_split=0.2,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)


Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6274 - loss: 0.6225 - val_accuracy: 0.5944 - val_loss: 0.5892
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7223 - loss: 0.5766 - val_accuracy: 0.7622 - val_loss: 0.5270
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7838 - loss: 0.5393 - val_accuracy: 0.7832 - val_loss: 0.4981
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7663 - loss: 0.5137 - val_accuracy: 0.7832 - val_loss: 0.4896
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7838 - loss: 0.4943 - val_accuracy: 0.7622 - val_loss: 0.4812
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7891 - loss: 0.4907 - val_accuracy: 0.7622 - val_loss: 0.4882
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7750 - loss: 0.4836 - val_accuracy: 0.7832 - val_loss: 0.4956
Epoch 8/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7856 - loss: 0.4710 - val_accuracy: 0.7413 - v

In [10]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import json
import numpy as np


In [11]:
# Predictions
y_train_proba = final_model.predict(X_train).flatten()
y_test_proba = final_model.predict(X_test).flatten()

y_train_pred = (y_train_proba > 0.5).astype(int)
y_test_pred = (y_test_proba > 0.5).astype(int)

# Metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

cm = confusion_matrix(y_test, y_test_pred)


23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


In [12]:
results = {
    "model": "NeuralNetwork",
    "dataset": "Titanic",
    "best_params": best_params,

    "metrics": {
        "train_accuracy": float(train_accuracy),
        "test_accuracy": float(test_accuracy),
        "train_roc_auc": float(train_auc),
        "test_roc_auc": float(test_auc)
    },

    "confusion_matrix": {
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1])
    }
}


In [13]:
results_path = "/home/prateek/Prateek/LaunchPad/week6/Day4/src/Tunning/result.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {results_path}")


Results saved to /home/prateek/Prateek/LaunchPad/week6/Day4/src/Tunning/result.json


In [14]:
import os

models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

h5_path = os.path.join(models_dir, "final_model.h5")

final_model.save(h5_path)

print(f"Model saved to {h5_path}")


Model saved to models/final_model.h5
